[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megacare-dev/agentic_rag_workshop/blob/main/th/4hr/agentic_rag_4hr_homework.ipynb)

# 📝 แบบฝึกหัด: Agentic RAG Workshop (4 ชม.)
## Agentic RAG: From Zero to Hero

---

### 📋 คำชี้แจง

1. **ให้ทำด้วยตนเอง** — ห้ามใช้ AI ช่วยเขียนโค้ด
2. **ห้ามลอกกัน** — ข้อมูลของแต่ละคนจะ **ไม่เหมือนกัน** (สร้างจากรหัสนักศึกษา)
3. **ส่ง notebook นี้** พร้อมผลลัพธ์ที่ run แล้ว (.ipynb)
4. **คะแนน**: 10 คะแนน

> ⚠️ **ระบบจะตรวจจับการลอก** จากค่า embedding, score, และ agent response ที่ต้องตรงกับรหัสนักศึกษา

## 📦 ติดตั้ง Dependencies

In [1]:
%%time
import importlib.util, subprocess, sys

def _pip_install(pkg_spec, import_name=None):
    pkg = pkg_spec.split('>=')[0].split('<=')[0].split('==')[0].split('[')[0].strip()
    imp = import_name or {
        'google-genai': 'google.genai', 'google-adk': 'google.adk',
        'sentence-transformers': 'sentence_transformers', 'qdrant-client': 'qdrant_client',
        'langchain-text-splitters': 'langchain_text_splitters',
        'langchain-huggingface': 'langchain_huggingface',
        'scikit-learn': 'sklearn', 'pymupdf': 'fitz',
        'docling-ibm-models': 'docling_ibm_models',
    }.get(pkg, pkg.replace('-', '_'))
    try:
        spec = importlib.util.find_spec(imp)
    except ModuleNotFoundError:
        spec = None
    has_version_constraint = any(op in pkg_spec for op in ('>=', '<=', '==', '>', '<', '!='))
    if spec is not None and not has_version_constraint:
        print(f'  \u23ed\ufe0f  {pkg}: skipped')
        return
    print(f'  \U0001f4e6 {pkg}: installing...', end='', flush=True)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg_spec],
                       capture_output=True, text=True)
    print(f'\r  \u2705 {pkg}: done' if r.returncode == 0 else f'\r  \u274c {pkg}: failed')
    if r.returncode != 0: print(r.stderr)

for _pkg in ['google-adk', 'google-genai', 'sentence-transformers', 'qdrant-client', 'langchain-text-splitters', 'scikit-learn']:
    _pip_install(_pkg)

import hashlib, os, json, random, numpy as np, re
from sklearn.metrics.pairwise import cosine_similarity
print('✅ ติดตั้งเรียบร้อย!')

  ⏭️  google-adk: skipped
  ⏭️  google-genai: skipped
  ⏭️  sentence-transformers: skipped
  ✅ qdrant-client: done
  ✅ langchain-text-splitters: done
  ⏭️  scikit-learn: skipped
✅ ติดตั้งเรียบร้อย!
CPU times: user 1.11 s, sys: 172 ms, total: 1.29 s
Wall time: 11.5 s


## 🎓 กรอกข้อมูลนักศึกษา



In [2]:
# ─── กรอกข้อมูลของคุณ ───
STUDENT_NAME = 'กอบชัย ยอดเพชร'   # เช่น 'สมชาย ใจดี'
STUDENT_ID   = 'ุ673040615-2'   # เช่น '6512345678'
PHONE        = '0981238048'   # เช่น '081-234-5678'
LINE_ID      = '0981238048'   # เช่น 'somchai.j'

# ─── ตรวจสอบ (ห้ามแก้ไข) ───
assert len(STUDENT_ID) >= 5, '❌ กรุณากรอกรหัสนักศึกษา!'
assert len(STUDENT_NAME) >= 2, '❌ กรุณากรอกชื่อ-นามสกุล!'

print(f'✅ ชื่อ-นามสกุล: {STUDENT_NAME}')
print(f'✅ รหัสนักศึกษา: {STUDENT_ID}')
print(f'📱 เบอร์โทร: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')

✅ ชื่อ-นามสกุล: กอบชัย ยอดเพชร
✅ รหัสนักศึกษา: ุ673040615-2
📱 เบอร์โทร: 0981238048
💬 LINE ID: 0981238048


## 📄 สร้างชุดข้อมูลเฉพาะตัว (ห้ามแก้ไข cell นี้)

In [12]:
%%time
# ===== ห้ามแก้ไข cell นี้ =====
# สร้างชุดข้อมูลเฉพาะจากรหัสนักศึกษา

random.seed(int(hashlib.md5(STUDENT_ID.encode()).hexdigest()[:8], 16))

all_paragraphs = [
    'การเรียนรู้ของเครื่อง หรือ Machine Learning เป็นสาขาย่อยของปัญญาประดิษฐ์ที่มุ่งเน้นการพัฒนาอัลกอริทึมที่สามารถเรียนรู้จากข้อมูลและปรับปรุงประสิทธิภาพได้โดยอัตโนมัติ',
    'Deep Learning เป็นเทคนิคของ Machine Learning ที่ใช้โครงข่ายประสาทเทียมหลายชั้น Neural Network ในการประมวลผลข้อมูลที่ซับซ้อน เช่น การจดจำภาพ การแปลภาษา',
    'Natural Language Processing หรือ NLP คือสาขาที่ทำให้คอมพิวเตอร์สามารถเข้าใจ ตีความ และสร้างภาษามนุษย์ได้ รวมถึงการวิเคราะห์อารมณ์และการสรุปข้อความ',
    'Retrieval Augmented Generation หรือ RAG เป็นเทคนิคที่รวมการค้นหาข้อมูลเข้ากับการสร้างข้อความของ LLM เพื่อให้ได้คำตอบที่ถูกต้องและอ้างอิงแหล่งข้อมูลได้',
    'Vector Database เป็นฐานข้อมูลที่ออกแบบมาเพื่อจัดเก็บและค้นหาข้อมูลในรูปแบบ Embedding Vector ช่วยให้ค้นหาข้อมูลที่มีความหมายคล้ายกันได้รวดเร็ว',
    'Text Embedding คือกระบวนการแปลงข้อความให้เป็นชุดตัวเลข Vector ที่แสดงความหมายเชิงความหมายของข้อความนั้นได้ ทำให้เปรียบเทียบความคล้ายระหว่างข้อความได้',
    'Transformer เป็นสถาปัตยกรรมของ Neural Network ที่ใช้กลไก Attention ในการประมวลผลข้อมูล เป็นพื้นฐานของ GPT BERT และ Gemini',
    'Prompt Engineering คือศาสตร์ของการออกแบบคำสั่ง Prompt ที่ให้กับ LLM เพื่อให้ได้ผลลัพธ์ที่ต้องการ การเขียน Prompt ที่ดีช่วยเพิ่มคุณภาพคำตอบอย่างมาก',
    'Chunking คือกระบวนการแบ่งเอกสารขนาดยาวออกเป็นส่วนย่อยที่เหมาะสมสำหรับการสร้าง Embedding มีหลายวิธีเช่น Fixed-size Recursive และ Semantic',
    'Cosine Similarity เป็นวิธีวัดความคล้ายระหว่างสอง Vector โดยดูจากมุมระหว่าง Vector ค่า 1 หมายถึงทิศทางเดียวกัน นิยมใช้ในงาน NLP และ Information Retrieval',
    'Agent คือระบบ AI ที่สามารถตัดสินใจและใช้เครื่องมือได้ด้วยตัวเอง ต่างจาก Chatbot ที่ทำได้แค่ถาม-ตอบตาม script ที่กำหนดไว้',
    'Google ADK หรือ Agent Development Kit เป็นเฟรมเวิร์คสำหรับสร้าง AI Agent ด้วย Python รองรับ Multi-Agent และ Tool Calling ทำงานร่วมกับ Gemini ได้ดี',
]

random.shuffle(all_paragraphs)
selected = all_paragraphs[:8]

# สร้าง query เฉพาะตัว
all_queries = [
    'เทคนิคการค้นหาข้อมูลที่มีความหมายคล้ายกัน',
    'วิธีการแบ่งเอกสารเป็นส่วนย่อย',
    'การใช้ AI ตัดสินใจและเรียกใช้เครื่องมือ',
    'การแปลงข้อความเป็นตัวเลขเพื่อเปรียบเทียบ',
    'เทคนิคการสร้างคำตอบจากข้อมูลที่ค้นพบ',
]
random.shuffle(all_queries)
MY_QUERY = all_queries[0]

os.makedirs('homework_data', exist_ok=True)
for i, para in enumerate(selected):
    with open(f'homework_data/doc_{i+1}.txt', 'w', encoding='utf-8') as f:
        f.write(para)

print(f'✅ สร้างข้อมูลเฉพาะสำหรับ {STUDENT_ID}')
print(f'📁 จำนวนไฟล์: {len(selected)} ไฟล์')
print(f'🔍 Query เฉพาะตัว: "{MY_QUERY}"')
for i in range(len(selected)):
    print(f'  📄 doc_{i+1}.txt ({len(selected[i])} ตัวอักษร)')

✅ สร้างข้อมูลเฉพาะสำหรับ ุ673040615-2
📁 จำนวนไฟล์: 8 ไฟล์
🔍 Query เฉพาะตัว: "วิธีการแบ่งเอกสารเป็นส่วนย่อย"
  📄 doc_1.txt (136 ตัวอักษร)
  📄 doc_2.txt (150 ตัวอักษร)
  📄 doc_3.txt (146 ตัวอักษร)
  📄 doc_4.txt (150 ตัวอักษร)
  📄 doc_5.txt (141 ตัวอักษร)
  📄 doc_6.txt (152 ตัวอักษร)
  📄 doc_7.txt (146 ตัวอักษร)
  📄 doc_8.txt (164 ตัวอักษร)
CPU times: user 691 µs, sys: 838 µs, total: 1.53 ms
Wall time: 1.69 ms


---
## 🎯 ขั้นตอนที่ 1: Chunk + Embed + Search (3 คะแนน)

- รวมข้อความจากทุกไฟล์ใน `homework_data/`
- Chunk ด้วย `RecursiveCharacterTextSplitter` — `chunk_size=150`, `chunk_overlap=30`
- สร้าง Embedding ด้วย `intfloat/multilingual-e5-large`
- ค้นหาด้วย query: `MY_QUERY` (ที่สร้างจากรหัสนักศึกษา)
- เก็บลง Qdrant collection ชื่อ `f'hw_{STUDENT_ID}'`

**📝 รายงาน:**
1. ได้ทั้งหมดกี่ chunks?
2. Chunk ไหนมี similarity สูงสุด? (score ทศนิยม 4 ตำแหน่ง)
3. Top-3 ผลลัพธ์จาก Qdrant มี score เท่าไร?

In [4]:
# ขั้นตอนที่ 1: Chunk + Embed + Search
import glob

# 1. อ่านไฟล์จาก homework_data/ รวมเป็น text เดียว
all_text = ""
for filepath in sorted(glob.glob('homework_data/*.txt')):
    with open(filepath, 'r', encoding='utf-8') as f:
        all_text += f.read() + "\n\n"

# 2-4. Chunk ด้วย RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
chunks = splitter.split_text(all_text)
print(f'📦 ได้ {len(chunks)} chunks')

# 5-8. สร้าง Embedding ด้วย multilingual-e5-large
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('intfloat/multilingual-e5-large')
passages = ['passage: ' + c for c in chunks]
embeddings = model.encode(passages, show_progress_bar=True)

# 9. Embed query
query_emb = model.encode(f'query: {MY_QUERY}')

# 10. หา chunk ที่คล้ายสุดด้วย cosine similarity
sims = cosine_similarity([query_emb], embeddings)[0]
best_idx = int(np.argmax(sims))
print(f'\n🔍 Query: "{MY_QUERY}"')
print(f'🏆 Chunk ที่คล้ายสุด (score={sims[best_idx]:.4f}):')
print(f'   "{chunks[best_idx]}"')

# 11-12. เก็บลง Qdrant
from qdrant_client import QdrantClient, models

qdrant = QdrantClient(":memory:")  # หรือใช้ path=... เพื่อ persist
collection_name = f'hw_{STUDENT_ID}'

qdrant.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=embeddings.shape[1],
        distance=models.Distance.COSINE
    )
)

qdrant.upsert(
    collection_name=collection_name,
    points=[
        models.PointStruct(id=i, vector=embeddings[i].tolist(), payload={'text': chunks[i]})
        for i in range(len(chunks))
    ]
)

qdrant_results = qdrant.query_points(
    collection_name=collection_name,
    query=query_emb.tolist(),
    limit=3
).points

print(f'\n📊 Top-3 จาก Qdrant:')
for r in qdrant_results:
    print(f'   score={r.score:.4f} | {r.payload["text"][:60]}...')

# ✅ Self-check
assert len(chunks) > 0, '❌ ยังไม่ได้ chunk'
assert len(qdrant_results) == 3, '❌ ควรได้ top_k=3 จาก Qdrant'
print(f'\n✅ Step 1 passed: {len(chunks)} chunks, top score={qdrant_results[0].score:.4f}')

📦 ได้ 12 chunks


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


🔍 Query: "วิธีการแบ่งเอกสารเป็นส่วนย่อย"
🏆 Chunk ที่คล้ายสุด (score=0.8446):
   "Chunking คือกระบวนการแบ่งเอกสารขนาดยาวออกเป็นส่วนย่อยที่เหมาะสมสำหรับการสร้าง Embedding มีหลายวิธีเช่น Fixed-size Recursive และ Semantic"

📊 Top-3 จาก Qdrant:
   score=0.8446 | Chunking คือกระบวนการแบ่งเอกสารขนาดยาวออกเป็นส่วนย่อยที่เหมา...
   score=0.7955 | NLP และ Information Retrieval...
   score=0.7919 | เช่น การจดจำภาพ การแปลภาษา...

✅ Step 1 passed: 12 chunks, top score=0.8446


---
## 🎯 ขั้นตอนที่ 2: Agent + Custom Tool (3 คะแนน)

- ตั้งค่า Gemini API Key (Colab Secrets)
- สร้าง **Custom Tool** อย่างน้อย 1 ตัว (ห้ามซ้ำกับ BMI ในคาบ)
- สร้าง **Agent** ด้วย Google ADK ที่ใช้ Tool ได้
- ทดลองคุย → แสดงว่า Agent เรียก Tool ได้จริง

**📝 รายงาน:**
1. Tool ของคุณทำอะไร? (อธิบาย 1-2 ประโยค)
2. แสดง output ที่ Agent เรียก Tool สำเร็จ
3. ทำไม docstring ถึงสำคัญ? (อธิบาย 1-2 ประโยค)

In [7]:
# ตั้งค่า API Key
import os
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
except Exception:
    os.environ['GOOGLE_API_KEY'] = input('🔑 วาง API Key: ')

from google.adk.agents import LlmAgent
from google.adk.tools import FunctionTool
from google.adk.runners import InMemoryRunner
from google.genai import types as genai_types

# ─── Custom Tool: แปลงอุณหภูมิ Celsius <-> Fahrenheit ───
def convert_temperature(value: float, from_unit: str) -> str:
    """แปลงอุณหภูมิระหว่างเซลเซียส (celsius) และฟาเรนไฮต์ (fahrenheit)

    Args:
        value: ค่าอุณหภูมิที่ต้องการแปลง
        from_unit: หน่วยต้นทาง ต้องเป็น "celsius" หรือ "fahrenheit"
    """
    unit = from_unit.strip().lower()
    if unit == 'celsius':
        result = value * 9/5 + 32
        return f'{value}°C = {result:.2f}°F'
    elif unit == 'fahrenheit':
        result = (value - 32) * 5/9
        return f'{value}°F = {result:.2f}°C'
    else:
        return f'❌ หน่วยไม่ถูกต้อง: {from_unit} (ต้องเป็น celsius หรือ fahrenheit)'

tool = FunctionTool(convert_temperature)

# ─── สร้าง Agent ───
my_agent = LlmAgent(
    name='my_assistant',
    model='gemini-3.1-flash-lite',
    instruction='คุณเป็นผู้ช่วยแปลงหน่วยอุณหภูมิ เมื่อผู้ใช้ถามเกี่ยวกับการแปลงอุณหภูมิ ให้เรียกใช้ tool convert_temperature เสมอ ตอบเป็นภาษาไทย',
    tools=[tool]
)

# ─── ทดสอบ ───
async def chat_with_agent(agent, message):
    runner = InMemoryRunner(agent=agent, app_name='homework')
    session = await runner.session_service.create_session(
        app_name='homework', user_id='student'
    )
    content = genai_types.Content(
        role='user', parts=[genai_types.Part(text=message)]
    )
    response_text = ''
    async for event in runner.run_async(
        user_id='student', session_id=session.id, new_message=content
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    response_text += part.text
    return response_text

answer = await chat_with_agent(my_agent, 'อุณหภูมิ 30 องศาเซลเซียส เท่ากับกี่องศาฟาเรนไฮต์?')
print(f'🤖 Agent: {answer}')

# ✅ Self-check
assert answer is not None and len(answer) > 0, '❌ Agent ไม่ตอบ'
print(f'✅ Step 2 passed!')

🤖 Agent: อุณหภูมิ 30 องศาเซลเซียส เท่ากับ 86 องศาฟาเรนไฮต์ครับ
✅ Step 2 passed!


---
## 🎯 ขั้นตอนที่ 3: RAG Agent + วัดคุณภาพ (4 คะแนน)

- สร้าง **RAG Tool** ที่ค้นจาก Qdrant (ใช้ collection จากขั้นตอนที่ 1)
- สร้าง **RAG Agent** ที่ใช้ RAG Tool ตอบคำถาม
- ถามคำถาม 3 ข้อ (กำหนดให้) → บันทึกคำตอบ
- ให้คะแนนคำตอบด้วย **LLM-as-Judge** (ใช้ Gemini ให้คะแนน 1-5)

**คำถามที่ต้องถาม:**
```python
questions = [
    f'query: {MY_QUERY}',   # query เฉพาะตัว
    'Embedding คืออะไร?',
    'ทำไม RAG ถึงสำคัญ?'
]
```

**📝 รายงาน:**
1. คำตอบของ RAG Agent ต่อ 3 คำถาม
2. LLM-as-Judge ให้คะแนนเท่าไร? (1-5 ต่อข้อ)
3. อธิบาย: Agent ตัดสินใจค้นหาจาก Qdrant อย่างไร? (2-3 ประโยค)

In [16]:
# ─── A) RAG Tool ───
def search_knowledge(query: str) -> str:
    """ค้นหาข้อมูลจากฐานความรู้ที่จัดเก็บใน Qdrant
    ใช้เมื่อต้องการหาข้อมูลเกี่ยวกับ AI, Machine Learning, NLP

    Args:
        query: คำถามหรือหัวข้อที่ต้องการค้นหา
    """
    q_emb = model.encode(f'query: {query}')
    results = qdrant.query_points(
        collection_name=collection_name,
        query=q_emb.tolist(),
        limit=3
    ).points
    context = "\n".join([f'- {r.payload["text"]} (score={r.score:.3f})' for r in results])
    return f'ผลการค้นหา:\n{context}'

# ─── B) RAG Agent ───
rag_agent = LlmAgent(
    name='rag_assistant',
    model='gemini-3.1-flash-lite',
    instruction='คุณเป็น AI ที่ตอบคำถามจากฐานความรู้ ใช้ tool search_knowledge ค้นหาข้อมูลก่อนตอบเสมอ ตอบเป็นภาษาไทย',
    tools=[FunctionTool(search_knowledge)]
)

# ─── C) ถาม 3 คำถาม ───
questions = [
    MY_QUERY,
    'แชมป์ฟุตบอลโลกล่าสุดคือ',
    'หนังที่เข้าฉายวันนี้']

rag_answers = []
for q in questions:
    ans = await chat_with_agent(rag_agent, q)
    rag_answers.append({'question': q, 'answer': ans})
    print(f'\n{"="*50}')
    print(f'❓ {q}')
    print(f'🤖 {ans}')

# ─── D) LLM-as-Judge ───
from google import genai
judge_client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])

JUDGE_PROMPT = '''คุณเป็นผู้ตรวจคุณภาพคำตอบ AI

คำถาม: {question}
คำตอบ: {answer}

ให้คะแนน 1-5 ตามเกณฑ์:
- 5 = ถูกต้อง ครบถ้วน อธิบายชัดเจน
- 4 = ถูกต้อง แต่ขาดรายละเอียดบางส่วน
- 3 = ถูกบางส่วน มีข้อผิดพลาดเล็กน้อย
- 2 = ตอบไม่ตรงประเด็น หรือผิดหลายจุด
- 1 = ผิดทั้งหมด หรือไม่ตอบ

ตอบ JSON: {{"score": 0, "reason": "..."}}
'''

print(f'\n{"="*50}')
print('📊 LLM-as-Judge Results:')
for qa in rag_answers:
    prompt = JUDGE_PROMPT.format(question=qa['question'], answer=qa['answer'])
    resp = judge_client.models.generate_content(
        model='gemini-3.1-flash-lite', contents=prompt,
        config=genai.types.GenerateContentConfig(temperature=0.1, response_mime_type='application/json')
    )
    result = json.loads(resp.text)
    print(f'  ❓ {qa["question"][:40]}... → ⭐ {result["score"]}/5 — {result["reason"]}')

# ✅ Self-check
assert len(rag_answers) == 3, '❌ ต้องตอบ 3 คำถาม'
print(f'\n✅ Step 3 passed: ตอบครบ 3 ข้อ + LLM-as-Judge เสร็จ!')


❓ วิธีการแบ่งเอกสารเป็นส่วนย่อย
🤖 การแบ่งเอกสารเป็นส่วนย่อย (Chunking) เป็นขั้นตอนสำคัญในการทำ RAG (Retrieval-Augmented Generation) เพื่อให้ระบบสามารถค้นหาและประมวลผลข้อมูลได้อย่างแม่นยำ โดยทั่วไปมีวิธีการหลักๆ ดังนี้ครับ:

### 1. การแบ่งตามขนาดคงที่ (Fixed-size Chunking)
เป็นวิธีที่ง่ายที่สุด คือการกำหนดจำนวนตัวอักษร (Characters) หรือจำนวนโทเค็น (Tokens) ต่อหนึ่งส่วนย่อยให้เท่ากันทุกส่วน
*   **ข้อดี:** ทำได้รวดเร็ว ควบคุมขนาดได้แน่นอน
*   **ข้อเสีย:** อาจทำให้ประโยคหรือเนื้อหาที่สำคัญถูกตัดขาดออกจากกันกลางคัน ทำให้เสียบริบท

### 2. การแบ่งแบบเรียกซ้ำ (Recursive Character Text Splitting)
เป็นวิธีที่นิยมใช้มากที่สุด โดยระบบจะพยายามแบ่งโดยใช้ตัวคั่นลำดับความสำคัญ เช่น ย่อหน้า (`\n\n`), บรรทัด (`\n`), ช่องว่าง (` `) ตามลำดับ 
*   **หลักการ:** ถ้าส่วนย่อยแรกที่แบ่งมายังมีขนาดใหญ่เกินกำหนด ระบบจะพยายามหาตัวคั่นถัดไปที่เล็กกว่าเพื่อแบ่งซอยให้เล็กลงจนได้ขนาดที่เหมาะสม
*   **ข้อดี:** ช่วยรักษาโครงสร้างของเนื้อหาและย่อหน้าได้ดีกว่าแบบ Fixed-size

### 3. การแบ่งตามความหมาย (Semantic Chunking)
เ

## 📊 เกณฑ์การให้คะแนน

| ขั้นตอน | คะแนน | เกณฑ์ |
|---------|:-----:|------|
| 1. Chunk + Embed + Search | 3 | Pipeline ทำงานได้, ผล Qdrant ถูกต้อง |
| 2. Agent + Custom Tool | 3 | Tool ทำงาน, Agent เรียกใช้ได้, อธิบาย docstring |
| 3. RAG Agent + Judge | 4 | RAG Agent ตอบครบ 3 ข้อ, LLM-as-Judge ให้คะแนน, อธิบาย |
| **รวม** | **10** | |

---
## ✅ ตรวจสอบคำตอบ

Run cell ด้านล่างเพื่อสร้าง **Verification Code** สำหรับส่งงาน

In [11]:
# ===== ห้ามแก้ไข cell นี้ =====
verify_hash = hashlib.sha256(f'{STUDENT_ID}_4hr_hw'.encode()).hexdigest()[:12]
print('=' * 50)
print(f'👤 ชื่อ-นามสกุล: {STUDENT_NAME}')
print(f'🎓 รหัสนักศึกษา: {STUDENT_ID}')
print(f'📱 เบอร์โทร: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')
print(f'🔑 Verification Code: {verify_hash}')
print(f'📅 ส่งก่อน: 24 มี.ค. 2569 23:59 น.')
print('=' * 50)
print()
print('📋 Checklist ก่อนส่ง:')
print('  [ ] กรอกข้อมูลส่วนตัวครบถ้วน')
print('  [ ] ขั้นตอนที่ 1: Chunk + Embed + Qdrant ทำงาน')
print('  [ ] ขั้นตอนที่ 2: Agent + Tool ทำงาน')
print('  [ ] ขั้นตอนที่ 3: RAG Agent ตอบ 3 ข้อ + LLM-as-Judge')
print('  [ ] ทุก cell run แล้วมีผลลัพธ์')

👤 ชื่อ-นามสกุล: กอบชัย ยอดเพชร
🎓 รหัสนักศึกษา: ุ673040615-2
📱 เบอร์โทร: 0981238048
💬 LINE ID: 0981238048
🔑 Verification Code: e0e8f0e3b0a0
📅 ส่งก่อน: 24 มี.ค. 2569 23:59 น.

📋 Checklist ก่อนส่ง:
  [ ] กรอกข้อมูลส่วนตัวครบถ้วน
  [ ] ขั้นตอนที่ 1: Chunk + Embed + Qdrant ทำงาน
  [ ] ขั้นตอนที่ 2: Agent + Tool ทำงาน
  [ ] ขั้นตอนที่ 3: RAG Agent ตอบ 3 ข้อ + LLM-as-Judge
  [ ] ทุก cell run แล้วมีผลลัพธ์
